# Load Contoso Coffee into the lakehouse

Phase 2 of the Power BI AI demo.

**Before you run this:**
1. Attach this notebook to the `LH_ContosoCoffee` lakehouse (Explorer pane, Add lakehouse).
2. Upload the four CSVs from the repo `data/` folder to `Files/raw/` in that lakehouse.

**The GitHub Copilot version of this phase:** delete the code cells below, keep this
markdown cell, and ask GitHub Copilot Chat to write the loader from the description.
Then compare what it wrote to what is here. That is the more interesting demo.

**What it must do:**
- Read `dim_date.csv`, `dim_product.csv`, `dim_store.csv`, `fact_sales.csv` from `Files/raw/`
- Apply an explicit schema. Do not let money land as strings and do not let dates land as strings.
  Every downstream Copilot failure in this demo traces back to a bad data type.
- Write each one as a managed delta table with the same name, overwriting if present
- Print row counts so the load can be verified against known values

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DateType, DecimalType, BooleanType
)

RAW = "Files/raw"

money = DecimalType(18, 2)

SCHEMAS = {
    "dim_date": StructType([
        StructField("date_key", DateType(), False),
        StructField("year", IntegerType(), False),
        StructField("quarter", StringType(), False),
        StructField("month_number", IntegerType(), False),
        StructField("month_name", StringType(), False),
        StructField("year_month", StringType(), False),
        StructField("day_of_week", StringType(), False),
        StructField("day_of_week_number", IntegerType(), False),
        StructField("is_weekend", BooleanType(), False),
    ]),
    "dim_product": StructType([
        StructField("product_key", IntegerType(), False),
        StructField("product_name", StringType(), False),
        StructField("category", StringType(), False),
        StructField("subcategory", StringType(), False),
        StructField("unit_price", money, False),
        StructField("unit_cost", money, False),
    ]),
    "dim_store": StructType([
        StructField("store_key", IntegerType(), False),
        StructField("store_name", StringType(), False),
        StructField("city", StringType(), False),
        StructField("state", StringType(), False),
        StructField("region", StringType(), False),
        StructField("store_type", StringType(), False),
        StructField("opened_date", DateType(), False),
    ]),
    "fact_sales": StructType([
        StructField("sales_order_id", IntegerType(), False),
        StructField("date_key", DateType(), False),
        StructField("store_key", IntegerType(), False),
        StructField("product_key", IntegerType(), False),
        StructField("channel", StringType(), False),
        StructField("quantity", IntegerType(), False),
        StructField("gross_amount", money, False),
        StructField("discount_amount", money, False),
        StructField("net_amount", money, False),
        StructField("cost_amount", money, False),
    ]),
}

for table, schema in SCHEMAS.items():
    (
        spark.read.format("csv")
        .option("header", "true")
        .schema(schema)
        .load(f"{RAW}/{table}.csv")
        .write.mode("overwrite")
        .format("delta")
        .saveAsTable(table)
    )
    print(f"wrote {table}")

## Verify

These counts are exact. The data generator is seeded, so anyone who runs
`python data/generate_data.py` gets the same rows.

| Table | Expected rows |
| --- | --- |
| `dim_date` | 731 |
| `dim_product` | 12 |
| `dim_store` | 8 |
| `fact_sales` | 64,335 |

Total net revenue must be `412918.50`. If it is not, the load lost or duplicated rows,
or a money column landed as a string and got truncated.

In [ ]:
from decimal import Decimal

EXPECTED_ROWS = {"dim_date": 731, "dim_product": 12, "dim_store": 8, "fact_sales": 64335}
EXPECTED_NET_REVENUE = Decimal("412918.50")

ok = True
for table, expected in EXPECTED_ROWS.items():
    actual = spark.table(table).count()
    flag = "OK  " if actual == expected else "FAIL"
    ok = ok and actual == expected
    print(f"{flag} {table:<14} expected {expected:>6}  actual {actual:>6}")

net = spark.sql("SELECT SUM(net_amount) AS net FROM fact_sales").collect()[0]["net"]
net_ok = net == EXPECTED_NET_REVENUE
ok = ok and net_ok
print(f"\n{'OK  ' if net_ok else 'FAIL'} total net revenue expected {EXPECTED_NET_REVENUE}  actual {net}")
print("\nload verified" if ok else "\nload FAILED, do not continue to phase 3")

## Next

Phase 3. Build the semantic model. See `docs/03-model.md`.

From this lakehouse you can select `New semantic model` on the ribbon to create a Direct
Lake model over these four tables, then add the relationships and the measures from
`semantic-model/measures.dax`.